# Validate original models against test data from Ciara

There isn't enough human-transcribed data to train against, but we can use it for validating the AI transcription models

This notebook documents the validation against the original models with no fuine-tuning.


## 1. Model checkpoints to use

We will use 5 different models as-downloaded

- smolvlm2 - HuggingFaceTB/SmolVLM2-2.2B-Instruct
- granite4 - ibm-granite/granite-vision-4.1-4b
- gemma3 - google/gemma-3-4b-it
- gemma4 - google/gemma-4-E4B-it
- ministral - mistralai/Mistral-Small-3.1-24B-Instruct-2503

The human-transcribed test cases are in 

- test_data/from_Ciara/images


## 2. Run the extractions

Each command submits one extraction job to Azure. Run them in the terminal. They will complete quickly (the test dataset has only 64 images).

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --model smolvlm2 --images-path test_data/from_Ciara/images --transcriptions-path documents/DailyRainfall_UK/test_data/from_Ciara/transcriptions_0 --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --model granite4 --images-path test_data/from_Ciara/images --transcriptions-path documents/DailyRainfall_UK/test_data/from_Ciara/transcriptions_0 --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --model gemma3 --images-path test_data/from_Ciara/images --transcriptions-path documents/DailyRainfall_UK/test_data/from_Ciara/transcriptions_0 --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --model gemma4 --images-path test_data/from_Ciara/images --transcriptions-path documents/DailyRainfall_UK/test_data/from_Ciara/transcriptions_0 --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --model ministral --images-path test_data/from_Ciara/images --transcriptions-path documents/DailyRainfall_UK/test_data/from_Ciara/transcriptions_0 --total-shards 1 --batch-size 15 --extraction-registry outputs/extraction_registry.json extract 
```


## 4. Download the extractions from Azure

Get the extraction run_names from the extraction registry. In this case, they were:

- SmolVLM: 20260616-142852
- Granite4: 20260616-144739
- Gemma3: 20260616-145253
- Gemma4: 20260616-145335
- Ministral: 20260616-145351

Then download them in turn

```bash
bash scripts/aml_download.sh --run-name 20260616-142852
bash scripts/aml_download.sh --run-name 20260616-144739
bash scripts/aml_download.sh --run-name 20260616-145253
bash scripts/aml_download.sh --run-name 20260616-145335
bash scripts/aml_download.sh --run-name 20260616-145351
```




## 5. Make consensus from the downloaded extractions

Looks for cases where multiple extractions agree and flags those sections as reliable.

Setup - needed for the first run only.
```bash
mkdir outputs/consensus_Ciara_data
ln -s test_data/from_Ciara/images outputs/consensus_Ciara_data
```

Make the consensus and the validation plots.

```bash
bash scripts/run_consensus_pipeline.sh --dataset-root outputs/consensus_Ciara_data --variant-name consensus_0th_order --threshold 3 --null-threshold 5 --precision 3 --extraction-dir outputs/extractions/20260616-142852 --extraction-dir outputs/extractions/20260616-144739 --extraction-dir outputs/extractions/20260616-145253 --extraction-dir outputs/extractions/20260616-145335 --extraction-dir outputs/extractions/20260616-145351 --validate --ground-truth-dir test_data/from_Ciara/transcriptions --validation-sample-denominator 1
```

This will produce a consensus in ```outputs/consensus_Ciara_data/consensus_0th_order```; with transcriptions, summary and validation figures for all of the cases.